# KubeMedic GRPO Training on Hugging Face

This notebook keeps the workflow simple:
- load `HF_TOKEN` from `.env` when available
- install the KubeMedic package plus training extras
- smoke-test the deployed Space
- run the single-session GRPO trainer
- inspect the saved artifacts


In [1]:
import os
import subprocess
import sys
from pathlib import Path

os.environ.setdefault('WANDB_DISABLED', 'true')
os.environ.setdefault('TRL_EXPERIMENTAL_SILENCE', '1')

env_candidates = [Path.cwd() / '.env', Path.cwd().parent / '.env']
env_path = next((path for path in env_candidates if path.exists()), None)
if env_path:
    for line in env_path.read_text().splitlines():
        if line.startswith('HF_TOKEN=') and not os.environ.get('HF_TOKEN'):
            os.environ['HF_TOKEN'] = line.split('=', 1)[1].strip().strip('"').strip("'")
            break

if not os.environ.get('HF_TOKEN'):
    try:
        from google.colab import userdata
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    except Exception:
        pass

if not os.environ.get('HF_TOKEN'):
    try:
        os.environ['HF_TOKEN'] = subprocess.check_output(['hf', 'auth', 'token'], text=True).strip()
    except Exception:
        pass

SPACE_ID = 'ashiqabdulkhader/Kubemedic'
ENV_URL = f"https://{SPACE_ID.replace('/', '-')}.hf.space"
PACKAGE_INSTALL = f"openenv-kubemedic[train] @ git+https://huggingface.co/spaces/{SPACE_ID}.git"
OUTPUT_DIR = Path('outputs/kubemedic-qwen25-3b-grpo')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_EPISODES = 8
EVAL_EPISODES = 2
NUM_GENERATIONS = 2
MAX_COMPLETION_LENGTH = 256
MAX_TURNS = 8

print('HF token loaded:', bool(os.environ.get('HF_TOKEN')))
print('ENV_URL        :', ENV_URL)
print('PACKAGE_INSTALL:', PACKAGE_INSTALL)
print('OUTPUT_DIR     :', OUTPUT_DIR)
print('Kernel Python  :', sys.executable)
print('NUM_GENERATIONS:', NUM_GENERATIONS)


FileNotFoundError: Could not locate training.py from notebook working directory

In [ ]:
%pip install -q -U "{PACKAGE_INSTALL}" "wandb>=0.17.0"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
from Kubemedic import KubemedicAction, KubemedicEnv

env = KubemedicEnv(base_url=ENV_URL).sync()
try:
    result = env.reset(scenario='KUBE-03')
    print(result.observation)
    step_result = env.step(KubemedicAction(tool='kubectl_get', args={'resource': 'pods', 'namespace': 'challenge'}))
    print('\n--- smoke tool call ---\n')
    print(step_result.observation)
finally:
    env.close()


done=False reward=0.0 metadata={} t=0 scenario='KUBE-03' pods=[PodObservation(name='api-gw-59cb9f4bd-b94rz', namespace='challenge', phase='Running', reason=None, node='aks-userpool-41566868-vmss000001', restarts=0, priority_class=None), PodObservation(name='api-gw-59cb9f4bd-nf6dx', namespace='challenge', phase='Running', reason=None, node='aks-userpool-41566868-vmss000001', restarts=0, priority_class=None), PodObservation(name='auth-svc-75d997fbf6-m7r26', namespace='challenge', phase='Running', reason=None, node='aks-userpool-41566868-vmss000001', restarts=0, priority_class=None), PodObservation(name='order-svc-5b6f494cb-8xh5h', namespace='challenge', phase='Running', reason=None, node='aks-userpool-41566868-vmss000001', restarts=0, priority_class=None), PodObservation(name='payment-svc-765c658dc8-dkcrg', namespace='challenge', phase='Running', reason='CrashLoopBackOff', node='aks-userpool-41566868-vmss000001', restarts=1, priority_class=None)] nodes=[NodeObservation(name='aks-agentpoo

In [ ]:
!{sys.executable} -m Kubemedic.training \
  --env-url "{ENV_URL}" \
  --output-dir "{OUTPUT_DIR}" \
  --train-episodes {TRAIN_EPISODES} \
  --eval-episodes {EVAL_EPISODES} \
  --num-generations {NUM_GENERATIONS} \
  --per-device-train-batch-size {NUM_GENERATIONS} \
  --max-completion-length {MAX_COMPLETION_LENGTH} \
  --max-turns {MAX_TURNS} \
  --vllm-mode colocate


/usr/bin/python3: No module named training


In [ ]:
import json
import pandas as pd
from IPython.display import Image, display

reward_csv = OUTPUT_DIR / 'episode_rewards.csv'
summary_json = OUTPUT_DIR / 'training_summary.json'
reward_plot = OUTPUT_DIR / 'reward_plot.png'

if reward_csv.exists():
    display(pd.read_csv(reward_csv).tail(10))
else:
    raise FileNotFoundError(
        f'{reward_csv} was not created. Check the training cell output above for the real failure.'
    )

if summary_json.exists():
    print(json.loads(summary_json.read_text()))
if reward_plot.exists():
    display(Image(filename=str(reward_plot)))

trainer_plot = OUTPUT_DIR / 'trainer_metrics.png'
if trainer_plot.exists():
    display(Image(filename=str(trainer_plot)))


FileNotFoundError: outputs/kubemedic-qwen25-3b-grpo/episode_rewards.csv was not created. Check the training cell output above for the real failure.